In [1]:
# RCF + oil purification system


from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import feed_parameters, prices
from lignin_saf.systems.rcf import create_rcf_system
from lignin_saf.cellulosic_tea import create_cellulosic_ethanol_tea
from lignin_saf.systems.rcf_oil_purification import create_rcf_oil_purification_system
from lignin_saf.systems.monomer_purification import create_monomer_purification_system


from biosteam import main_flowsheet as F
import biosteam as bst

chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 541.7

chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('Poplar_In',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d', price=prices['Feedstock'])


rcf_system = create_rcf_system(ins=poplar_in)
rcf_system.simulate()


rcf_oil_purification_sys = create_rcf_oil_purification_system(ins=F.RCF_CRUDE_OUT)
rcf_oil_purification_sys.simulate()


monomer_purification_sys = create_monomer_purification_system(ins=F.PURE_OIL_OUT)
monomer_purification_sys.simulate()

WWT = bst.create_conventional_wastewater_treatment_system('WWT', ins=(F.WW_10, F.WastePulp, F.RCF_WW_OUTS, F.WW_11))


for unit in WWT.units:
    if hasattr(unit, 'strict_moisture_content'):
        unit.strict_moisture_content = False   # ← toggle here
    # To adjust the target moisture fraction (default 0.79 from Humbird):
    # if hasattr(unit, 'moisture_content'):
    #     unit.moisture_content = 0.6

BT = bst.facilities.BoilerTurbogenerator('BT', fuel_price=prices['CH4'])

gas_mixer= bst.Mixer('MIX_BT_gas',    ins=(WWT.outs[0], F.RCF_PSAWASTE_OUTS))

oligomers = F.WW_12
solids_to_BT = bst.Mixer('MIX_BT_solids', ins=(WWT.outs[1], oligomers), rigorous = True)



BT.ins[0] = solids_to_BT.outs[0]   # Connecting sludge and oligomers/dimers to BT solids feed
BT.ins[1] = gas_mixer.outs[0]   # Connecting biogas from WW treatment and PSA waste gases from RCF




rcf_monomers_system = bst.System(
    'Solo RCF system',
    path=(rcf_system, rcf_oil_purification_sys, monomer_purification_sys,WWT),
    facilities=[gas_mixer, solids_to_BT, BT],
)
rcf_monomers_system.simulate()

integrated_tea = create_cellulosic_ethanol_tea(rcf_monomers_system)

print(f'The MSP for RCF crude oil is  {round(integrated_tea.solve_price(F.MON_MONOMERS_OUT), 3)} USD/kg')




c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\_pump.py:224: RuntimeWarning: <Pump: RCF_PUMP1> no pump type available at current power (2.34e+03 hp), head (3.2e+03 ft), kinematic viscosity (5.83e-07 m2/s), and NPSH (4.18 ft); assuming centrigugal pump
  warn(f'{repr(self)} no pump type available at current power '
c:\users\hwadg\onedrive - the pennsylvania state university\shi_wadgama_shared\models\atjspk\lignin_saf\ligsaf_units.py:411: CostWarning: <SolvolysisReactor: RCF_RXR1> Vertical vessel length (58.75 ft) is out of bounds (12 to 40 ft) for cost correlation
  self._vertical_vessel_design(
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\_unit.py:1241: CostWarning: <IsentropicCompressor: RCF_COMP1> power (1.48

The MSP for RCF crude oil is  9.778 USD/kg


In [2]:
import thermosteam as tmo
#  Code just to increase the number of display units for the various components
tmo.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
tmo.MultiStream.display_units.N = 40  
bst.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
bst.MultiStream.display_units.N = 40  

In [3]:
F.WW_11

Stream: WW_11 from <LiquidsSplitCentrifuge: CENT303> to <Mixer: M601>
phase: 'l', T: 341.8 K, P: 101325 Pa
flow (kmol/hr): Water   0.152
                Hexane  38.9


In [4]:
F.WWT.outs[1]

Stream: sludge from <SludgeCentrifuge: S603> to <Mixer: MIX_BT_solids>
phase: 'l', T: 303.68 K, P: 101325 Pa
flow (kmol/hr): Water           3.03e+03
                AceticAcid      0.022
                Extract         0.00313
                NaOH            0.0508
                SolubleLignin   3.72e-08
                Glucan          18.1
                Xylan           4.53
                Arabinan        0.726
                Mannan          9.12
                Galactan        3.45
                WWTsludge       102
                Methanol        0.603
                Hexane          0.000362
                EthylAcetate    0.000126
                Propylguaiacol  5.97e-10
                Propylsyringol  5.06e-10
                Syringaresinol  9.04e-12
                G_Dimer         9.42e-10
                S_Oligomer      1.19e-10
                G_Oligomer      1.35e-10


In [5]:
F.WW_12

Stream: WW_12 from <MultiStageMixerSettlers: LLE300> to <Mixer: MIX_BT_solids>
phase: 'l', T: 311.87 K, P: 101325 Pa
flow (kmol/hr): Water           5.66e+03
                Propylguaiacol  0.00805
                Propylsyringol  0.00682
                Syringaresinol  4
                G_Dimer         4.62
                S_Oligomer      3.48
                G_Oligomer      3.94


In [6]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:
bst.nbtutorial()

In [ ]:
rcf_monomers_system.diagram()